# Module 12 Lab - Ethics, Fairness, and Bias in ML**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.**Sources of Bias:***   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

df = pd.read_csv(
    url,
    header=None,
    names=columns,
    sep=r',\s*',
    engine='python',
    na_values='?'
)

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model
model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

Overall model accuracy: 84.61%


## Part 2: Auditing the Model for FairnessHigh overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.**Concept: Group Fairness**One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [7]:
# --- ENTER YOUR CODE HERE ---

def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""

    # 1. Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)


# 4. Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(
    model, X_test, y_test, 'sex', 'Male'
)

acc_female = get_subgroup_accuracy(
    model, X_test, y_test, 'sex', 'Female'
)

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%


### Task 2: Deeper Dive with a Confusion MatrixAccuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [8]:
from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)

    tn, fp, fn, tp = confusion_matrix(
        y_subgroup, y_pred_subgroup
    ).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    return fpr, fnr


# --- ENTER YOUR CODE HERE ---

# 1. Calculate the rates for males and females
fpr_male, fnr_male = get_rates(
    model, X_test, y_test, 'sex', 'Male'
)

fpr_female, fnr_female = get_rates(
    model, X_test, y_test, 'sex', 'Female'
)

print(
    f"Male - False Positive Rate: {fpr_male:.2%}, "
    f"False Negative Rate: {fnr_male:.2%}"
)

print(
    f"Female - False Positive Rate: {fpr_female:.2%}, "
    f"False Negative Rate: {fnr_female:.2%}"
)

Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%


## 📝 Reflective Knowledge Check
**Instructions:** Answer the following questions in this markdown cell. Your answers should be based on **your specific results** from the code you ran above.
1.  **Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females? Which group does the model perform better for?
2.  **Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error (predicting high income when it's not)? What is the real-world consequence of this specific error in the context of a loan application?
3.  **Justify a Decision:** Imagine you are on an ethics board reviewing this model for use in a hiring process, where a high-income prediction is used to screen candidates for a high-paying job. Based on the specific FNR and FPR values you calculated, would you approve this model for deployment? Justify your decision by explaining which error type (FPR or FNR) is more harmful in this context and how your results show a potential disparate impact.
4.  **Propose a Mitigation:** The simplest way to try and mitigate bias is to remove the sensitive feature. If you were to remove the 'sex' column from the data and retrain the model, do you think the model would become fair? Why or why not? (Hint: Think about what other columns might be correlated with 'sex').


**[
  1. **Analyze Your Results:** Looking at the subgroup accuracies I calculated, there is a noticeable enough difference in how the model performs for males versus females. Based on the accuracy results of 81.20% for males, in comparison to 91.81% for females, the model performs better overall for females. There is a difference of about 10.61% points between the subgroups.This also shows why looking only at the overall accuracy of 84.61% can be misleading because it does not show the differences in performance between groups.

2.  **Interpret the Errors:** The False Positive Rate was 10.26% for males and only 2.81% for females.The model is more likely to make a False Positive error for males, making the model more likely to incorrectly predict that a male earns a high income of over $50,000 when in reality, he does not. In the context of a real world consequence of this specific error, this could cause a lender to view a male applicant as having a higher income than they may actually have, resulting in the person being potentially approved for a loan amount that they may not be able to afford.


3. **Justify a Decision:**  If I were on an ethics board, I would not approve this model for deployment. In a hiring process I believe the False Negative Rate would be more harmful because it means the model wrongly identifies a qualified high income person as belonging to the lower income group, possibly preventing that person from being a qualifying candidate for a high paying job. The False Negative Rate was 47.84% for females compared to 37.80% for males. Meaning women who actually earn above $50,000 are more likely to be misclassified as low income. Even though the model's overall accuracy is higher for females, their much higher FNR shows that accuracy alone does not guarantee fairness in this hiring scenario. This specific error could create a disadvantage for female candidates during hiring, it’s an evident risk of disparate impact. Until the model’s error imbalance is properly addressed, it should not be used in a real hiring process.

4. **Propose a Mitigation:** Removing the sex column might mitigate direct bias, but I do not think it would automatically make the model fair. Other columns, such as occupation and marital status, etc. could be correlated with sex and indirectly allow the model to learn gender linked patterns. Because of these correlations, the model could still produce unequal outcomes without directly knowing a person's sex. I would retrain the model without the sex column and then repeat the same fairness tests again to determine whether the differences in accuracy, False Positive Rates, and False Negative Rates actually improved. ]**



